# Lab 3 — Before/After Redesign
### SDA-DSC-112 · Module 3 · Colour, Annotation, and Decluttering · **50 minutes**

| | |
|---|---|
| **Objective** | Take a deliberately cluttered, mis-coloured Tayseer chart and redesign it: correct palette, decluttered, annotated with the takeaway, colour-blind-safe, and shipped in Arabic — then measure the improvement |
| **Dataset** | `tayseer_services.csv` · `brand_palette.csv` · `dim_regions.csv` (Arabic names) · `dim_calendar.csv` (Arabic months) |
| **Output** | A genuine before/after pair, a CVD simulation, an Arabic RTL edition, and a data-ink count |

| Task | Minutes |
|---|---|
| 1 · Study the cluttered chart and list every violation | 5 |
| 2 · Rebuild with the correct palette type | 10 |
| 3 · Apply the decluttering ladder | 10 |
| 4 · Annotate: takeaway title + exactly one callout | 10 |
| 5 · Colour-blind simulation, and the redundant cue | 5 |
| 6 · The Arabic RTL edition | 5 |
| 7 · Measure it: data-ink before vs after | 5 |

### Setup

The subject is **cost per transaction by region, latest month** — a measure with a
*meaningful midpoint* (the 18 SAR programme target). Remember which weight it takes:
`SUM(cost × transactions) / SUM(transactions)`, never `AVERAGE(cost)`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import tayseer_viz as tv
tv.apply_theme()

df = tv.load_facts()
regions = tv.load("dim_regions.csv")
calendar = tv.load("dim_calendar.csv", parse_dates=["month"])
brand = tv.load("brand_palette.csv")

latest = df[df.month == df.month.max()]
cost = (tv.by(latest, "region", "cost_per_txn_sar")
          .rename(columns={"cost_per_txn_sar": "cost_sar"}))
print(f"{len(cost)} regions · cost per transaction "
      f"{cost.cost_sar.min():.1f}–{cost.cost_sar.max():.1f} SAR · target {tv.TARGET_COST:.0f} SAR")

# Your figures go to out/starter/ so they never overwrite the reference set.
tv.set_out_dir(tv.OUT / "starter")

---
## Task 1 — Study the cluttered chart and list every violation *(5 min)*

`cluttered_m3.png` is rebuilt here in code so the "before" is reproducible and every
sin is visible in the source. Study it, then write the violation list.

The audit target, built in code so every sin is visible in the source:

In [ ]:
def build_cluttered(cost, path="lab3_before_cluttered"):
    """The 'before'. Nine deliberate violations — name them all in Task 1."""
    with plt.rc_context({"axes.grid": True, "axes.spines.top": True,
                         "axes.spines.right": True, "axes.spines.left": True,
                         "axes.titlelocation": "center", "axes.titleweight": "normal"}):
        fig, ax = plt.subplots(figsize=(10, 6), facecolor="#F3F0E7")
        d = cost.sort_values("region")                       # violation: alphabetical
        rainbow = plt.get_cmap("gist_rainbow")(np.linspace(0, .92, len(d)))

        # fake 3D: a dark offset shadow bar behind each bar
        ax.bar(np.arange(len(d)) + .09, d.cost_sar, width=.62, color="#7a7a7a",
               zorder=1)                                      # violation: pseudo-3D
        bars = ax.bar(np.arange(len(d)), d.cost_sar, width=.62, color=rainbow,
                      edgecolor="black", linewidth=1.6, zorder=2)

        # red/green as the ONLY signal of above/below target
        for bar, v in zip(bars, d.cost_sar):
            bar.set_facecolor("#D62728" if v > tv.TARGET_COST else "#2CA02C")

        ax.set_ylim(12, 26)                                   # violation: truncated axis
        ax.set_xticks(np.arange(len(d)))
        ax.set_xticklabels(d.region, rotation=55, ha="right")
        ax.set_ylabel("Cost per transaction (SAR)")
        ax.set_xlabel("Region")                               # violation: redundant label
        ax.grid(color="#8888FF", lw=1.3, alpha=.9)            # violation: heavy gridlines
        ax.set_facecolor("#EFEFEF")
        ax.legend([bars[0], bars[1]],
                  ["Above target (bad)", "Below target (good)"],
                  loc="center", fontsize=11)                  # violation: legend over data
        ax.set_title("Cost per Transaction by Region")        # violation: topic title
    print(tv.save_fig(fig, path))
    return fig

_ = build_cluttered(cost)
plt.show()

In [ ]:
# --- TODO -----------------------------------------------------------------
# Build `violations`: columns ['class', 'violation', 'why_it_fails', 'fix'].
# Classes to cover: Colour, Accessibility, Chart junk, Honesty, Layout, Argument.
# The package's expected output finds 9; there are 12 if you are thorough.
violations = None
# --------------------------------------------------------------------------

if violations is None:
    print("TODO not implemented yet — expected: >= 9 rows across >= 5 classes.")
    print("Hints: what palette type does ORDERED data with a TARGET need?")
    print("       what is the only cue distinguishing above-target from below?")
    print("       where does the Y axis start?")
else:
    assert list(violations.columns) == ["class", "violation", "why_it_fails", "fix"]
    assert len(violations) >= 9 and violations["class"].nunique() >= 5
    display(violations)
    print("ok —", len(violations), "violations")

---
## Task 2 — Rebuild with the correct palette type *(10 min)*

Three palette families, and the data type chooses:

| Data type | Palette | Tayseer example |
|---|---|---|
| Categorical (regions, channels) | Distinct hues, ≤ 7 | channel = app / web / branch |
| Sequential (0 → high) | One hue, light → dark | adoption 0–100% |
| **Deviation from a midpoint** | **Diverging, centred on the midpoint** | **cost vs the 18 SAR target** |

Cost-vs-target is a *deviation* measure. The neutral colour must sit on the **target**,
not on the data mean — otherwise the palette hides the one boundary that matters.

In [ ]:
display(brand[brand.palette_type.str.contains("Diverging")][
    ["token", "hex", "name", "colour_blind_safe", "usage_rule"]])

s = cost.sort_values("cost_sar")

# --- TODO -----------------------------------------------------------------
# Two panels side by side:
#   LEFT  the WRONG palette (a sequential ramp on deviation data)
#   RIGHT the RIGHT palette:
#         norm = mcolors.TwoSlopeNorm(vmin=..., vcenter=tv.TARGET_COST, vmax=...)
#         cmap = plt.get_cmap(tv.DIVERGING)
#         colours = [cmap(norm(v)) for v in s.cost_sar]
# Both panels: x from 0, sorted, no y gridlines. Save to lab3_palette_choice.png.
# Then print which regions exceed the target.
over = None
# --------------------------------------------------------------------------

if over is None:
    print("TODO not implemented yet — expected 5 regions above 18 SAR:")
    print("  Jazan, Najran, Al-Baha, Asir, Northern Borders")
else:
    assert len(over) == 5 and set(over.region) == set(tv.STALLED)
    print("ok —", ", ".join(over.region))

---
## Task 3 — Apply the decluttering ladder *(10 min)*

In order. Junk first, then lighten, and **never** remove data ink:

1. Remove chart junk (3D, gradients, shadows, background fills)
2. Remove or lighten gridlines
3. Remove the plot border / drop spines
4. Remove redundant axes and ticks; label directly
5. Remove the legend (direct-label instead)
6. Mute everything that is context to grey; reserve saturation for the signal

In [ ]:
# --- TODO -----------------------------------------------------------------
# Write `rung(ax, level, cost)` that draws the SAME chart at each rung of the ladder,
# then render rungs 0-6 as a grid of panels and save to lab3_declutter_ladder.png.
#   level 0  the 'before'          level 4  redundant labels/ticks removed
#   level 1  junk removed          level 5  legend replaced by colour + position
#   level 2  gridlines lightened   level 6  context muted to grey + target line
#   level 3  spines dropped
# Rule: you may never remove a BAR. Bars are data ink.
rung = None
# --------------------------------------------------------------------------

if rung is None:
    print("TODO not implemented yet — expected a 7-panel ladder, out/starter/lab3_declutter_ladder.png")
else:
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    for i, ax in enumerate(axes.ravel()):
        if i < 7:
            rung(ax, i, cost)
        else:
            ax.axis("off")
    fig.tight_layout(); print(tv.save_fig(fig, "lab3_declutter_ladder")); plt.show()

---
## Task 4 — Annotate: takeaway title + exactly one callout *(10 min)*

An unannotated chart says "here is some data". An annotated chart says "here is what it
means". **One argument, one callout.** The instinct to annotate everything recreates the
clutter you just removed.

This is the "after" of the before/after pair.

In [ ]:
d = cost.sort_values("cost_sar")

# --- TODO -----------------------------------------------------------------
# Build the 'after'. Requirements:
#   * tv.sorted_bar with highlight=tv.STALLED, reference=tv.TARGET_COST
#   * a TAKEAWAY title (states the conclusion)
#   * EXACTLY ONE ax.annotate() callout, pointing at the mark that carries the argument
#   * optionally one subtitle line via fig.text
# Save to lab3_after_annotated.png.
AFTER_DONE = False
# --------------------------------------------------------------------------

if not AFTER_DONE:
    print("TODO not implemented yet. The callout must state the ARGUMENT, e.g.")
    print("  'the same five regions that stalled on adoption' — not 'cost in SAR'.")
    print("  Count your annotations: exactly one.")

---
## Task 5 — Colour-blind simulation, and the redundant cue *(5 min)*

`cvd_sim.py` in the course assets is reproduced here. It converts a saved chart to
linear RGB, applies the Machado (2009) severity-1.0 CVD matrices, and returns the image
as a protanope or deuteranope sees it.

Roughly **8% of men** have red–green colour vision deficiency. The rule is not "avoid
red and green" — it is **never rely on colour alone**.

In [ ]:
# The simulator is given to you — the exercise is running it and reading the result.
CVD = {
    "protanopia": np.array([[0.152286,  1.052583, -0.204868],
                            [0.114503,  0.786281,  0.099216],
                            [-0.003882, -0.048116, 1.051998]]),
    "deuteranopia": np.array([[0.367322,  0.860646, -0.227968],
                              [0.280085,  0.672501,  0.047413],
                              [-0.011820, 0.042940,  0.968881]]),
}

def _to_linear(c):
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

def _to_srgb(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.0031308, c * 12.92, 1.055 * c ** (1 / 2.4) - 0.055)

def cvd_sim(rgb, kind="deuteranopia"):
    lin = _to_linear(rgb[..., :3])
    return _to_srgb(lin @ CVD[kind].T)

# --- TODO -----------------------------------------------------------------
# 1. Read out/starter/lab3_before_cluttered.png and your out/starter/lab3_after_annotated.png
#    with plt.imread.
# 2. Render a 2x3 grid: before/after x (normal, deuteranopia, protanopia).
# 3. Build `redundant_cues`: every cue in your AFTER chart that carries the message
#    WITHOUT hue. Columns ['cue', 'implementation', 'survives_cvd', 'why'].
redundant_cues = None
# --------------------------------------------------------------------------

if redundant_cues is None:
    print("TODO not implemented yet — expected >= 3 non-hue cues.")
    print("  Ask: if I printed this in greyscale, is the message still readable?")
else:
    assert len(redundant_cues) >= 3
    display(redundant_cues)

---
## Task 6 — The Arabic RTL edition *(5 min)*

For a Saudi executive audience this is not polish. Three things must be right:

1. **Glyph shaping.** matplotlib draws Arabic as isolated, left-to-right letters unless
   you run `arabic_reshaper` + `python-bidi` first and use an Arabic-capable font.
   Broken Arabic is worse than English-only.
2. **Reading order.** A ranking axis should put the first/best on the **right**, and a
   **time axis must run right-to-left** (earliest on the right). A left-to-right time
   axis reads backwards to an Arabic-first audience.
3. **Numerals.** Choose Western Arabic (0123) or Arabic-Indic (٠١٢٣) deliberately and
   consistently. Saudi government analytics decks commonly use Western Arabic numerals;
   confirm the house style.

In [ ]:
ok, font = tv.setup_arabic()
print(f"Arabic shaping available: {ok}   font: {font}")

ar_name = dict(zip(regions.region, regions.region_ar))
AK = tv.arabic_kwargs()

# --- TODO -----------------------------------------------------------------
# Build the Arabic edition. Three things must be right:
#   1. every Arabic string goes through tv.ar(...) before it reaches matplotlib,
#      and every text call gets **AK so it uses the Arabic font
#   2. the ranking axis puts the BEST value on the RIGHT; the value axis moves to
#      the right (ax.yaxis.tick_right()); titles use loc="right"
#   3. a time axis must run right-to-left: ax.invert_xaxis()
# If `ok` is False, fall back to English labels AND say so on the chart — never
# ship disconnected Arabic letters or tofu boxes.
ARABIC_DONE = False
# --------------------------------------------------------------------------

if not ARABIC_DONE:
    print("TODO not implemented yet — expected out/starter/lab3_arabic_rtl.png with:")
    print("  joined Arabic glyphs, value axis on the right, time axis inverted.")
    print("Self-check: does tv.ar('نسبة التبني') differ from the raw string? ->",
          tv.ar("نسبة التبني") != "نسبة التبني")

---
## Task 7 — Measure it: data-ink before vs after *(5 min)*

Tufte's **data-ink ratio** = data ink ÷ total ink. Count elements, not pixels: every
bar is data ink; every gridline, spine, tick, legend frame, shadow and background fill is
not. A ≥ 2× reduction in non-data elements is the Module 3 benchmark.

In [ ]:
# --- TODO -----------------------------------------------------------------
# 1. Write count_ink(fig) -> {'data_ink', 'non_data_ink', 'total', 'ratio'}.
#    Data ink  : bars (ax.patches), solid data lines
#    Non-data  : visible spines, gridlines, dashed reference lines, legend,
#                axis labels, non-white facecolors, tick marks
# 2. Count the BEFORE figure and your AFTER figure.
# 3. Build `comparison` and assert the non-data-ink reduction is at least 2x.
# 4. Render the side-by-side BEFORE / AFTER pair and save it.
comparison = None
# --------------------------------------------------------------------------

if comparison is None:
    print("TODO not implemented yet — the Module 3 benchmark is >= 2x fewer")
    print("  non-data elements, with the same number of bars. You may not delete a bar.")
else:
    display(comparison)

---
## Lab 3 checklist

- [ ] Task 1 — ≥ 9 violations found, sorted into colour / accessibility / junk / honesty
- [ ] Task 2 — diverging palette **centred on 18 SAR**, not on the data mean
- [ ] Task 3 — ladder applied in order; no bar was deleted
- [ ] Task 4 — takeaway title and **exactly one** callout
- [ ] Task 5 — CVD simulation run; ≥ 3 non-hue cues carry the message
- [ ] Task 6 — Arabic glyphs joined, value axis right, time axis inverted
- [ ] Task 7 — ≥ 2× fewer non-data elements, same number of bars

Save your work as `lab3_solution.ipynb`.